# AAROH - Phase 2: Mother-Tongue Translation Layer

This notebook demonstrates:
1. **IndicTrans2 / SageMaker Architecture**: High-accuracy translation for Indian languages.
2. **DynamoDB Key-Value Glossary**: Ultra-fast (< 2ms) pedagogical term lookups.
3. **Tribal Dialect Adaptations**: Preserving and injecting cultural anchors (Gondi, Santhali, Chhattisgarhi, Bhili).
4. **Full Lesson Translation**: Translating chunks extracted in Phase 1 into child-friendly mother tongue.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.aaroh.configuration.configuration import ConfigurationManager
from src.aaroh.components.translation import TranslationComponent
from src.aaroh.database.glossary_db import GlossaryDatabaseManager
import pandas as pd

## 1. Inspect Curated DynamoDB Glossary

In [ ]:
config_mgr = ConfigurationManager()
glossary_db = GlossaryDatabaseManager(config_mgr.get_dynamodb_config())
component = TranslationComponent(config_mgr.get_translation_config(), glossary_db)

# Display cached terms
terms_data = []
for ck, val in glossary_db.cache.items():
    terms_data.append({
        "Source Term": val["source_term"],
        "Standard Hindi": val["translated_term"],
        "Gondi (gon)": val.get("dialects", {}).get("gon", "-"),
        "Santhali (sat)": val.get("dialects", {}).get("sat", "-"),
        "Chhattisgarhi (hne)": val.get("dialects", {}).get("hne", "-")
    })

pd.DataFrame(terms_data)

## 2. Compare Standard Translation vs Tribal Dialect Adaptation

In [ ]:
sample_text = "Villagers collect cool water from the earthen pot and protect the check dam."

# 1. Standard Hindi
hi_res = component.translate_text(sample_text, source_lang="en", target_lang="hi")
print("Standard Hindi:")
print(hi_res.translated_text)
print(f"Latency: {hi_res.latency_ms} ms\n")

# 2. Gondi Dialect Adaptation
gon_res = component.translate_text(sample_text, source_lang="en", target_lang="hi", target_dialect="gon")
print("Gondi Dialect Adaptation:")
print(gon_res.translated_text)
print(f"Latency: {gon_res.latency_ms} ms")